In [1]:
# ============================================================
# GOVERNANCE SECTION GENERATOR
# IFRS S1/S2 governance | Azure OpenAI REST endpoint
# ============================================================

import os
import json
import re
import urllib.request
import urllib.error
from typing import TypedDict, Literal
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END

# ── ENV LOADING ──────────────────────────────────────────────
# In notebooks, .env is often not loaded because the kernel runs from a different folder.
# find_dotenv() makes the notebook search upward from the current path.
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_URL = os.getenv("AZURE_OPENAI_CHAT_URL")

if AZURE_OPENAI_CHAT_URL:
    AZURE_OPENAI_CHAT_URL = AZURE_OPENAI_CHAT_URL.strip().strip('"').strip("'")

if not AZURE_OPENAI_API_KEY:
    raise ValueError("Missing AZURE_OPENAI_API_KEY. Add it to .env or environment variables.")

if not AZURE_OPENAI_CHAT_URL:
    raise ValueError(
        "Missing AZURE_OPENAI_CHAT_URL. Expected full Azure URL, e.g.\n"
        "https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=2024-02-01"
    )

if not AZURE_OPENAI_CHAT_URL.startswith("https://"):
    raise ValueError(f"Invalid Azure URL: {AZURE_OPENAI_CHAT_URL!r}")

print("Azure OpenAI REST config loaded")
print("URL loaded:", AZURE_OPENAI_CHAT_URL[:80] + "...")
print("Key loaded:", True, "| length:", len(AZURE_OPENAI_API_KEY))


def _azure_chat_completion(messages: list[dict], temperature: float, max_tokens: int, json_mode: bool = False) -> dict:
    """Minimal Azure OpenAI REST call using the full URL stored in AZURE_OPENAI_CHAT_URL."""
    payload = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if json_mode:
        payload["response_format"] = {"type": "json_object"}

    req = urllib.request.Request(
        AZURE_OPENAI_CHAT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "api-key": AZURE_OPENAI_API_KEY,
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode(errors="replace")
        print("Azure HTTP error:", e.code)
        print(body[:1500])
        raise
    except urllib.error.URLError as e:
        print("Azure URL / connection error:", e)
        print("URL used:", repr(AZURE_OPENAI_CHAT_URL))
        raise


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    data = _azure_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=2200,
        json_mode=False,
    )
    return data["choices"][0]["message"]["content"]


def call_llm_json(system_prompt: str, user_prompt: str) -> dict:
    data = _azure_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=1600,
        json_mode=True,
    )
    return json.loads(data["choices"][0]["message"]["content"])

print("LLM helper functions ready")


Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Azure OpenAI REST config loaded
URL loaded: https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-4o-...
Key loaded: True | length: 32
LLM helper functions ready


In [2]:
# ── LOAD PAYLOAD ─────────────────────────────────────────────
# Works both locally and in this sandbox if the payload is placed next to the notebook.

candidate_paths = [
    Path("Data/payload_BANK01.json"),
    Path("payload_BANK01.json"),
    Path.cwd() / "Data" / "payload_BANK01.json",
    Path.cwd() / "payload_BANK01.json",
]

PAYLOAD_PATH = next((p for p in candidate_paths if p.exists()), None)
if PAYLOAD_PATH is None:
    raise FileNotFoundError(
        "Could not find payload_BANK01.json. Put it in Data/payload_BANK01.json "
        "or in the same folder as the notebook."
    )

with open(PAYLOAD_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

bank_name = payload["bank"]["bank_name"]
print(f"Loaded payload for: {bank_name}")
print(f"Payload path: {PAYLOAD_PATH}")


Loaded payload for: Eurolux Universal Bank AG
Payload path: payload_BANK01.json


In [3]:
# ── EVIDENCE EXTRACTOR ───────────────────────────────────────
# Pulls only the governance-relevant fields from the payload.
# Strict governance fixes added:
# - formal mandate / charter evidence is separated from activity evidence
# - board trade-off evidence is explicitly checked; if absent, the writer must state that limitation
# - management process evidence is converted into a process flow, not only an inventory
# - skills adequacy process evidence is separated from skills outcome metrics
# - assurance scope limitation is made explicit for financed emissions / Scope 3
# - meeting_id retained for board decision traceability


def _is_present(value) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "nan", "none", "null"}


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _normalise_text(value) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def extract_management_process_evidence(payload: dict, year: int = 2024) -> dict:
    """Summarise the process evidence needed for management responsibility."""
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    if not risks:
        return {
            "risk_register_available": False,
            "message": "No climate_risk_register records available for the reporting year.",
            "process_flow_instruction": (
                "Do not invent management process details. State that the available evidence does not "
                "include risk-register process records for the reporting year."
            )
        }

    frequencies = sorted({str(r.get("monitoring_frequency")) for r in risks if _is_present(r.get("monitoring_frequency"))})
    risk_categories = sorted({str(r.get("risk_category")) for r in risks if _is_present(r.get("risk_category"))})
    risk_ratings = sorted({str(r.get("risk_rating")) for r in risks if _is_present(r.get("risk_rating"))})
    scenario_links = sorted({str(r.get("scenario_analysis_link")) for r in risks if _is_present(r.get("scenario_analysis_link"))})
    mitigation_actions = sorted({str(r.get("mitigation_actions")) for r in risks if _is_present(r.get("mitigation_actions"))})

    integrated_count = sum(1 for r in risks if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks if r.get("changed_since_prior_period") is True)

    # Keep only the most useful examples for prompt compactness.
    material_risk_examples = []
    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    sorted_risks = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            float(r.get("financial_impact_meur") or 0)
        ),
        reverse=True,
    )
    for r in sorted_risks[:5]:
        material_risk_examples.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        })

    return {
        "risk_register_available": True,
        "reporting_year": year,
        "risk_count": len(risks),
        "erm_integrated_count": integrated_count,
        "changed_since_prior_period_count": changed_count,
        "monitoring_frequencies": frequencies,
        "risk_categories": risk_categories,
        "risk_ratings": risk_ratings,
        "scenario_analysis_links": scenario_links,
        "mitigation_actions": mitigation_actions[:8],
        "material_risk_examples": material_risk_examples,
        "process_flow_instruction": (
            "Write management responsibility as a process flow: identify climate risks in the climate risk register; "
            "classify them by category, time horizon and rating; monitor them at the recorded quarterly or semi-annual "
            "frequency; link relevant risks to scenario analysis where a scenario link exists; define mitigation actions; "
            "and use ERM integration to support management monitoring and board or committee review where required. "
            "Do not invent a formal escalation threshold unless explicitly provided."
        )
    }


def extract_governance_evidence(payload: dict) -> dict:

    gov_records = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})

    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict) and "reporting_year" in r
    }

    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    PRIORITY_TOPICS = [
        "scenario", "transition", "net_zero", "target", "carbon_credit",
        "remuneration", "tcfd", "green_finance", "physical_risk", "risk", "esg"
    ]

    def decision_score(m: dict) -> int:
        topics = str(m.get("climate_topics_discussed", "")).lower()
        decision = str(m.get("decision_summary", "")).lower()
        ifrs = str(m.get("ifrs_s2_para_evidence", "")).lower()
        score = sum(1 for t in PRIORITY_TOPICS if t in topics or t in decision)
        if "6(a)(v)" in ifrs:
            score += 2
        if str(m.get("committee_type", "")).lower() == "full_board":
            score += 1
        return score

    minutes_2024 = [
        m for m in board_minutes
        if isinstance(m, dict)
        and _safe_int(m.get("reporting_year")) == 2024
        and m.get("decision_made_flag") is True
        and _is_present(m.get("decision_summary"))
        and _is_present(m.get("meeting_id"))
    ]

    # Deduplicate by decision text only; keep the highest-scoring exact record and retain meeting_id.
    best_by_decision = {}
    for m in minutes_2024:
        decision_text = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        current = best_by_decision.get(decision_text)
        if current is None or decision_score(m) > decision_score(current):
            best_by_decision[decision_text] = m

    selected_decisions = []
    for m in sorted(best_by_decision.values(), key=decision_score, reverse=True)[:6]:
        selected_decisions.append({
            "meeting_id": m.get("meeting_id"),
            "date": m.get("meeting_date"),
            "committee": m.get("committee_name"),
            "committee_type": m.get("committee_type"),
            "topics_discussed": m.get("climate_topics_discussed"),
            "decision": m.get("decision_summary"),
            "ifrs_evidence_para": m.get("ifrs_s2_para_evidence"),
            "internal_ref": f"[REF:{m.get('meeting_id')}]",
        })

    gov_2024 = gov_by_year.get("2024", {})

    # Evidence gap assessment for strict governance disclosure.
    # These are intentionally conservative: the writer may state a limitation, but must not invent missing details.
    governance_instrument_fields = [
        "committee_charter", "committee_terms_of_reference", "board_mandate", "esg_committee_mandate",
        "formal_climate_mandate", "governance_policy_reference", "committee_charter_climate_mandate"
    ]
    formal_mandate_available = any(_is_present(gov_2024.get(f)) for f in governance_instrument_fields)

    tradeoff_terms = ["tradeoff", "trade-off", "capital allocation", "profitability", "cost", "risk appetite", "competing"]
    tradeoff_decisions = [
        m for m in minutes_2024
        if any(term in str(m.get("decision_summary", "")).lower() or term in str(m.get("climate_topics_discussed", "")).lower() for term in tradeoff_terms)
    ]
    board_tradeoff_evidence_available = len(tradeoff_decisions) > 0

    skills_process_fields = [
        "skills_matrix", "skills_assessment_process", "board_skills_review", "skills_adequacy_assessment",
        "director_training_frequency", "training_hours", "skills_gap_analysis"
    ]
    skills_adequacy_process_available = any(_is_present(gov_2024.get(f)) for f in skills_process_fields)

    assurance_scope = str(gov_2024.get("assurance_scope", ""))
    financed_emissions_2024 = reporting_kpis.get("financed_emissions_2024_tco2e")
    assurance_scope_limitation = {
        "assurance_scope": assurance_scope,
        "external_assurance": gov_2024.get("external_assurance"),
        "provider": gov_2024.get("assurance_provider"),
        "standard": gov_2024.get("assurance_standard"),
        "financed_emissions_2024_tco2e": financed_emissions_2024,
        "financed_emissions_in_scope": "financed" in assurance_scope.lower() or "scope 3" in assurance_scope.lower(),
        "instruction": (
            "State that assurance covers only the stated scope. If the stated scope is Scope 1 and 2 emissions, "
            "do not imply financed emissions or other Scope 3 categories are assured. For a bank, explicitly clarify "
            "that financed emissions are outside the stated assurance scope based on available evidence."
        )
    }

    return {
        "bank": {
            "name": bank.get("bank_name"),
            "country": bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": 2024,
        "comparative_years": [2022, 2023],
        "governance_2024": {
            "board_size": gov_2024.get("board_size"),
            "independent_directors_pct": gov_2024.get("independent_directors_pct"),
            "esg_committee_exists": gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked": gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct": gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct": gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency": gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name": gov_2024.get("management_committee_name"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "skills_development_programme": gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance": gov_2024.get("external_assurance"),
            "assurance_provider": gov_2024.get("assurance_provider"),
            "assurance_scope": gov_2024.get("assurance_scope"),
            "assurance_standard": gov_2024.get("assurance_standard"),
            "tcfd_aligned": gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned": gov_2024.get("ifrs_s2_aligned"),
        },
        "governance_trend": gov_trend,
        "management_process_evidence": extract_management_process_evidence(payload, year=2024),
        "board_decisions_2024": selected_decisions,
        "strict_governance_evidence": {
            "formal_governance_mandate_available": formal_mandate_available,
            "formal_governance_mandate_instruction": (
                "Do not claim the ESG & Sustainability Committee has a formal climate mandate unless charter/terms-of-reference evidence is provided. "
                "If no formal instrument is available, say the payload evidences committee activity and meeting frequency but does not include the committee charter or terms of reference."
            ),
            "board_tradeoff_evidence_available": board_tradeoff_evidence_available,
            "tradeoff_decisions": tradeoff_decisions[:3],
            "board_tradeoff_instruction": (
                "Discuss board trade-offs only if explicit trade-off evidence exists. If not, state that the board decision evidence identifies climate-related decisions, "
                "but does not describe specific trade-offs such as profitability, capital allocation, implementation cost, risk appetite or competing strategic priorities."
            ),
            "skills_adequacy_process_available": skills_adequacy_process_available,
            "skills_adequacy_instruction": (
                "Use the board climate expertise percentage and skills development programme as outcome/activity evidence. "
                "Do not invent a formal skills adequacy assessment process. If no skills assessment evidence exists, say the payload does not describe a formal board skills adequacy assessment process."
            ),
            "assurance_scope_limitation": assurance_scope_limitation,
        },
        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year "
                "where climate-related topics appeared on the agenda. It does NOT mean "
                "percentage of agenda time devoted to climate."
            ),
            "management_committee_names": (
                "Committee names are recorded by year only. The evidence does not prove that "
                "one committee evolved into, replaced, or was renamed as another. State the 2024 "
                "committee name and, if comparative names are used, present them neutrally."
            ),
            "board_decision_traceability": (
                "Each selected decision includes a meeting_id for audit traceability. Meeting IDs "
                "may be used internally but should not be printed in the final report unless required."
            ),
            "avoid_duplication": (
                "Do not list the same board decisions twice. Board oversight should summarise decision governance; "
                "the detailed dated list belongs only in the Board and committee decisions subsection."
            )
        }
    }


evidence = extract_governance_evidence(payload)

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Trend years: {[t['year'] for t in evidence['governance_trend']]}")
print(f"Risk-register records for management process: {evidence['management_process_evidence'].get('risk_count')}")
print("Strict governance evidence flags:")
for k, v in evidence["strict_governance_evidence"].items():
    if isinstance(v, bool):
        print(f"- {k}: {v}")
print("Selected decisions with internal refs:")
for d in evidence["board_decisions_2024"]:
    print(f"- {d['date']} | {d['committee']} | {d['decision']} | {d['internal_ref']}")


Evidence extracted for: Eurolux Universal Bank AG
Board decisions selected: 5
Trend years: [2022, 2023, 2024]
Risk-register records for management process: 8
Strict governance evidence flags:
- formal_governance_mandate_available: False
- board_tradeoff_evidence_available: False
- skills_adequacy_process_available: False
Selected decisions with internal refs:
- 2024-01-24 | Full Board | Approved 2024 ESG report for publication | [REF:MTG-BANK01-FB-2024-001]
- 2024-04-15 | ESG & Sustainability Committee | Approved climate scenario analysis methodology | [REF:MTG-BANK01-ESG-2024-011]
- 2024-04-15 | ESG & Sustainability Committee | Approved carbon credit procurement budget | [REF:MTG-BANK01-ESG-2024-010]
- 2024-10-07 | Full Board | Endorsed updated transition plan | [REF:MTG-BANK01-FB-2024-004]
- 2024-11-01 | Full Board | Endorsed net-zero interim target revision | [REF:MTG-BANK01-FB-2024-007]


In [4]:
# ── STATE DEFINITION ─────────────────────────────────────────
class GovernanceState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

In [5]:
# ── GOVERNANCE REQUIREMENTS ─────────────────────────────────
# IFRS references are used internally for coverage only.
# The final markdown headings and body must NOT include IFRS paragraph references.

IFRS_GOVERNANCE_REQUIREMENTS = """
STRICT GOVERNANCE DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Board oversight:
  - Describe board oversight of climate-related risks and opportunities.
  - Distinguish activity evidence from formal mandate evidence.
  - If committee charter / terms of reference / formal mandate evidence is not provided, state that the available evidence shows committee activity and meeting frequency, but does not include the formal governance instrument.
  - Explain how the board is informed about climate matters, using climate reporting cadence and board agenda evidence.
  - Explain how climate is considered in oversight of strategy, major transactions, risk management, metrics and targets.
  - Discuss board-level trade-offs only if evidenced. If no trade-off evidence exists, state that board decisions are evidenced but specific trade-offs are not described in the available minutes evidence.
  - Do not list the detailed board decisions here; summarise and point to the dedicated decisions subsection.

Management responsibility:
  - Which management body or role is responsible for climate-related risks and opportunities.
  - Write a process flow, not only an inventory: risk identification, register recording, classification, monitoring frequency, scenario links, mitigation actions, and ERM integration.
  - Be careful with escalation wording: only state a formal escalation threshold if explicit evidence exists. Otherwise use safe wording about board or committee review where required.

Climate skills and competencies:
  - Use board climate expertise and skills development programme evidence.
  - Describe the skills adequacy assessment process only if evidence exists.
  - If no skills adequacy process evidence exists, state that the payload evidences expertise percentage and skills development activity, but does not describe a formal skills adequacy assessment process.

Remuneration:
  - Explain whether and how climate-related performance metrics are incorporated into remuneration.
  - Cite percentage of CEO and all-executive remuneration linked to ESG/climate metrics where available.

Board and committee decisions:
  - Include the detailed dated list only in this subsection.
  - Use exact date, committee and decision pairings from board_decisions_2024.
  - Do not repeat the same detailed decisions in Board oversight.

External assurance and controls:
  - State assurance provider, standard, level and exact scope.
  - Explain limited assurance safely as lower assurance than reasonable assurance.
  - Do not imply assurance covers metrics outside the stated scope.
  - For a bank, if assurance covers only Scope 1 and Scope 2 emissions, clarify that financed emissions / Scope 3 are outside the stated assurance scope based on available evidence.
"""


In [6]:
# ── WRITER SYSTEM PROMPT ─────────────────────────────────────
WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section
of an IFRS S1/S2 aligned climate disclosure report for a commercial bank.

WRITING STANDARDS:
- Formal, third-person professional disclosure language suitable for publication.
- Specific and data-driven — cite exact figures, dates, and percentages.
- Every quantitative claim must come from the provided evidence — never invent numbers.
- Use IFRS requirements internally for coverage, but DO NOT put IFRS paragraph references in subsection headings or body text.
- No vague language such as "demonstrates commitment" unless backed by concrete evidence.
- Avoid overly strong assurance/control language such as "ensuring"; prefer "supporting", "providing", or "helping".
- Avoid compliance conclusions such as "aligned with IFRS S2 requirements", "fully aligned", "compliant", or "ensures reliability".
- Do not add a generic limitation disclaimer at the end.
- Specific evidence boundaries are allowed and required when evidence is missing, for example: "the available evidence does not describe...".
- Do not hedge when data is clearly available.

STRICT IFRS GOVERNANCE CONTROL:
- Distinguish board-level requirements from management-level requirements.
- Do not use management process evidence to satisfy board trade-off or board mandate disclosure.
- Board oversight must address mandate/activity, information flow, strategy/major transaction oversight, trade-offs or trade-off evidence limitation, and target/decision monitoring.
- Management responsibility must be written as a process flow, not as a raw inventory of risk register facts.
- Climate skills must separate outcome metrics from the process for assessing skills adequacy. Do not invent a skills adequacy process.
- External assurance must state the exact assurance scope and clarify when financed emissions / Scope 3 are outside that scope.
- Do not duplicate the detailed board decision list across multiple subsections.

HALLUCINATION CONTROL:
- Do not infer that a committee evolved, was renamed, replaced, strengthened, or specialized across years unless the evidence explicitly says so.
- If committee names differ by year, state only that different names are recorded across the comparative period.
- Do not create a transformation narrative from time-series values.
- Use board decision dates only from board_decisions_2024 and keep the date-decision-committee pairing exactly as provided.
- For escalation, do not write "significant or material climate risks are escalated" unless the evidence explicitly provides escalation criteria or route.
  Safer wording: "The risk register and ERM integration support management monitoring and board or committee review where required."
- For remuneration, avoid interpretive wording such as "demonstrates progressive integration". Use factual wording such as "indicates increased use".
- For assurance, explain limited assurance as lower than reasonable assurance. Do not call it "moderate assurance".

OUTPUT FORMAT:
Return markdown with exactly this subsection structure and NO IFRS paragraph references in headings or body text:

### Governance

#### Board oversight
...content...

#### Management responsibility
...content...

#### Climate skills and competencies
...content...

#### Remuneration and climate incentives
...content...

#### Board and committee decisions during 2024
...content...

#### External assurance and controls
...content...
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    is_revision = judge_feedback is not None

    base_instructions = f"""
BANK: {evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

GOVERNANCE REQUIREMENTS FOR THIS SECTION:
{IFRS_GOVERNANCE_REQUIREMENTS}

EVIDENCE (use ONLY this data):
{json.dumps(evidence, indent=2, ensure_ascii=False)}

CRITICAL INTERPRETATION RULES:
1. climate_on_board_agenda_pct = percentage of board MEETINGS where climate was on the agenda.
   NOT percentage of agenda time. Write it as: "climate featured on the agenda of X% of board meetings".
2. Committee names are evidence values by year only. Do NOT say "evolved from", "renamed from",
   "replaced", "progressively strengthened", or "specialized" unless explicit evidence says so.
   For BANK01, write: "In 2024, management-level climate governance is led by the Climate Risk Management Committee."
   If mentioning prior years, say only: "The recorded management committee name was X in 2022 and Y in 2023."
3. For board oversight, include a formal mandate sentence only if strict_governance_evidence.formal_governance_mandate_available is true.
   If false, state: "The available payload evidences ESG & Sustainability Committee activity and meeting frequency, but does not include the committee charter or terms of reference."
4. For board trade-offs, do not invent trade-offs. If strict_governance_evidence.board_tradeoff_evidence_available is false, state:
   "The available board minutes evidence identifies climate-related decisions but does not describe specific trade-offs considered by the board, such as profitability, capital allocation, implementation cost or risk appetite impacts."
5. For management responsibility, write a process flow using management_process_evidence:
   identify risks in the climate risk register; classify by category/time horizon/rating; monitor quarterly or semi-annually;
   link selected risks to scenario analysis; define mitigation actions; use ERM integration for management monitoring and board/committee review where required.
6. For climate skills, cite the board climate expertise trend and skills development programme.
   If strict_governance_evidence.skills_adequacy_process_available is false, state that the payload does not describe a formal board skills adequacy assessment process.
7. Include year-on-year trends for: board climate expertise %, CEO ESG compensation %, ESG committee meeting frequency, climate on board agenda %.
8. Board decisions: in Board oversight, only summarise that climate-related decisions are detailed below. Put the full dated list ONLY in "Board and committee decisions during 2024".
9. Cite at least 4 specific board/committee decisions from board_decisions_2024 with dates in the dedicated decision subsection.
   Use the exact date, committee, and decision pairing provided. Meeting IDs are internal traceability refs; do not print them.
10. For remuneration: cite both CEO ESG compensation % AND all-executive climate remuneration %.
    Prefer "indicates increased use" over "demonstrates progressive integration".
11. Explain limited assurance safely: limited assurance provides a lower level of assurance than reasonable assurance,
    based on procedures performed over the stated assurance scope. Do NOT say "moderate assurance".
12. Assurance scope: if stated scope is Scope 1 and 2 emissions, explicitly say financed emissions and other Scope 3 categories are outside the stated assurance scope based on available evidence.
13. Do NOT write "the bank's disclosures are aligned with IFRS S2 requirements". Safer wording:
    "The governance evidence is presented with reference to TCFD recommendations and IFRS governance disclosure requirements."
14. The final markdown must not contain visible IFRS paragraph references such as [IFRS S2 §6(a)], [IFRS S2 §7], §6(a), §6(b), or §7 in headings or body text.
"""

    if is_revision:
        return f"""
{base_instructions}

JUDGE FEEDBACK TO ADDRESS IN THIS REVISION:
{judge_feedback}

REVISION RULES:
- Fix every issue the judge flagged.
- Do not remove content that was not criticised.
- Do not add information not present in the evidence.
- Preserve the required subsection structure with NO visible IFRS paragraph references.

Write the revised governance section now.
""".strip()

    return f"""
{base_instructions}

Write the complete governance section now.
Follow the exact subsection structure specified in your instructions.
""".strip()


In [7]:
# ── JUDGE SYSTEM PROMPT ──────────────────────────────────────
JUDGE_SYSTEM = """
You are a strict IFRS S1/S2 compliance reviewer and ESG audit specialist.
Your job is to identify genuine gaps in a governance disclosure section — not to reward fluent writing.

SCORING ANCHOR:
  10: Perfect. Every requirement met, every figure cited, all trends present, board decisions specific and traceable, no unsupported interpretation.
  8-9: Strong. All subsections present, only very minor wording issues.
  6-7: Adequate draft. All subsections present but 2-3 content requirements thin or missing.
  4-5: Weak. Missing subsections or significant content gaps.
  1-3: Fails minimum disclosure requirements.

A score of 9 or 10 requires ALL of the following to be true:
- All 6 subsections present with substantive content.
- No visible IFRS paragraph references appear in subsection headings or body text.
- climate_on_board_agenda_pct correctly interpreted as meeting frequency, not agenda time.
- At least 4 distinct board/committee decisions cited with exact dates in the dedicated decisions subsection.
- Detailed board decisions are not duplicated in the Board oversight subsection.
- Year-on-year trends present for board expertise, CEO compensation, ESG committee meetings and climate agenda frequency.
- Both CEO ESG % and all-executive climate % cited in remuneration.
- Board oversight distinguishes committee activity from formal mandate. If no charter/terms-of-reference evidence exists, it states that boundary instead of inventing a mandate.
- Board oversight discusses board trade-offs OR states that the available board minutes evidence does not describe specific trade-offs.
- Management responsibility is written as a process flow: risk identification/register, classification, monitoring frequency, scenario links, mitigation actions and ERM integration. It must not read only like an inventory.
- Escalation wording is not overclaimed: formal escalation should not be stated unless explicit evidence exists.
- No unsupported committee evolution / renaming / strengthening narrative.
- Climate skills distinguishes outcome/activity evidence from a formal skills adequacy process. If no process evidence exists, it states that boundary.
- Limited assurance is explained safely as lower assurance than reasonable assurance.
- Assurance scope limitation is disclosed: if assurance scope is only Scope 1 and 2 emissions, financed emissions / Scope 3 are not implied to be assured.
- No generic limitation disclaimer at the end.
- No unsupported claims such as "ensures", "guarantees", "fully aligned", "fully resilient", "compliant", or "aligned with IFRS S2 requirements".

Score cannot exceed 8 if any of the above is false.
Score cannot exceed 6 if any required subsection is missing.
You must return valid JSON only — no other text.
""".strip()


def build_judge_prompt(draft: str, evidence: dict) -> str:

    gov_2024 = evidence.get("governance_2024", {})
    trend = evidence.get("governance_trend", [])
    decisions = evidence.get("board_decisions_2024", [])
    management_process = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})

    return f"""
Evaluate this governance section draft against the strict governance requirements.

DRAFT TO EVALUATE:
{draft}

KEY DATA AVAILABLE TO THE WRITER:
- Board size: {gov_2024.get('board_size')} members
- Independent directors: {gov_2024.get('independent_directors_pct')}%
- ESG committee meetings 2024: {gov_2024.get('esg_committee_meetings_per_year')}
- Board climate expertise: 2022={next((t['board_climate_expertise_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['board_climate_expertise_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('board_climate_expertise_pct')}%
- CEO ESG compensation: 2022={next((t['ceo_esg_compensation_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['ceo_esg_compensation_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('ceo_esg_compensation_pct')}%
- All-exec climate remuneration 2024: {gov_2024.get('all_exec_climate_remuneration_pct')}%
- Climate on board agenda: 2022={next((t['climate_on_board_agenda_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['climate_on_board_agenda_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('climate_on_board_agenda_pct')}%
- Management committee 2024: {gov_2024.get('management_committee_name')}
- Management process evidence: {json.dumps(management_process, ensure_ascii=False)}
- Strict governance evidence flags: {json.dumps(strict, ensure_ascii=False)}
- External assurance: {gov_2024.get('external_assurance')} by {gov_2024.get('assurance_provider')} under {gov_2024.get('assurance_standard')}; scope={gov_2024.get('assurance_scope')}
- Available board decisions with meeting IDs: {json.dumps(decisions, ensure_ascii=False)}

VERIFICATION CHECKLIST — answer each with true/false:
1. all_six_subsections_present: Are all 6 required subsections present?
2. no_visible_ifrs_refs: Are there NO visible IFRS paragraph references in headings or body text?
3. agenda_pct_correct: Is climate_on_board_agenda_pct described as meeting frequency, not agenda time?
4. four_distinct_decisions: Are at least 4 distinct board/committee decisions cited with exact dates in the dedicated decisions subsection?
5. no_duplicate_decisions: Are detailed dated decisions absent from Board oversight and listed only in the dedicated decisions subsection?
6. yoy_trends_present: Are year-on-year trends present for expertise, CEO compensation, ESG committee meetings, and climate agenda frequency?
7. both_remuneration_figures: Are both CEO ESG % and all-exec climate % cited?
8. formal_mandate_or_limitation: If no formal mandate evidence exists, does the text state that the evidence shows committee activity but not the formal charter/terms of reference? If formal mandate evidence exists, is it described?
9. board_tradeoffs_or_limitation: Does the text discuss board-level trade-offs if evidenced, or state that specific trade-offs are not described in the available minutes evidence?
10. management_process_flow: Does management responsibility read as a process flow, not merely an inventory of risks?
11. escalation_not_overclaimed: Does the section avoid formal escalation claims unless explicit evidence supports them?
12. no_unsupported_committee_evolution: No claim that committees evolved/renamed/replaced/strengthened unless explicitly evidenced?
13. skills_adequacy_or_limitation: Does the skills section describe a skills adequacy process if evidenced, or state that the payload only evidences expertise percentage and skills development activity if not?
14. assurance_explained_safely: Limited assurance explained without "moderate assurance" or overstated assurance conclusions?
15. assurance_scope_limitation: Does the assurance section state the exact scope and avoid implying financed emissions/Scope 3 are assured when the scope is only Scope 1 and 2?
16. no_limitation_disclaimer: No generic limitation disclaimer at the end? Specific evidence-boundary statements are allowed.
17. no_unsupported_strong_claims: No unsupported words/phrases like ensures, guarantees, fully aligned, compliant, aligned with IFRS S2 requirements?

COUNT how many checklist items are false.
Apply score ceiling:
- 0 false: score can reach 9-10
- 1 false: score cannot exceed 8
- 2 false: score cannot exceed 7
- 3+ false: score cannot exceed 6

Return this exact JSON structure:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approved": <true if overall_score >= 8 and 0 false checklist items, else false>,
  "checklist": {{
    "all_six_subsections_present": <true/false>,
    "no_visible_ifrs_refs": <true/false>,
    "agenda_pct_correct": <true/false>,
    "four_distinct_decisions": <true/false>,
    "no_duplicate_decisions": <true/false>,
    "yoy_trends_present": <true/false>,
    "both_remuneration_figures": <true/false>,
    "formal_mandate_or_limitation": <true/false>,
    "board_tradeoffs_or_limitation": <true/false>,
    "management_process_flow": <true/false>,
    "escalation_not_overclaimed": <true/false>,
    "no_unsupported_committee_evolution": <true/false>,
    "skills_adequacy_or_limitation": <true/false>,
    "assurance_explained_safely": <true/false>,
    "assurance_scope_limitation": <true/false>,
    "no_limitation_disclaimer": <true/false>,
    "no_unsupported_strong_claims": <true/false>,
    "false_count": <integer>
  }},
  "main_issues": [<list of specific issues found>],
  "required_fixes": [<specific actionable instructions for the reviser>]
}}
""".strip()


In [8]:
# ── DETERMINISTIC RULE CHECKS ────────────────────────────────
# These run before the LLM judge and can override it.
# They catch known failure modes from the governance evaluation.


def _section_text(draft: str, heading: str, next_headings: list[str]) -> str:
    """Extract text under a markdown h4 heading."""
    lower = draft.lower()
    h = heading.lower()
    start = lower.find(h)
    if start == -1:
        return ""
    start = start + len(h)
    end_candidates = []
    for nh in next_headings:
        pos = lower.find(nh.lower(), start)
        if pos != -1:
            end_candidates.append(pos)
    end = min(end_candidates) if end_candidates else len(draft)
    return draft[start:end]


def rule_check(draft: str) -> dict:
    text = draft.lower()

    required_subsections = [
        "#### board oversight",
        "#### management responsibility",
        "#### climate skills and competencies",
        "#### remuneration and climate incentives",
        "#### board and committee decisions",
        "#### external assurance",
    ]

    missing_subsections = [s for s in required_subsections if s not in text]

    unsupported_evolution_patterns = [
        "evolved from",
        "evolved into",
        "progressive strengthening",
        "progressively strengthening",
        "progressive integration",
        "demonstrates a progressive integration",
        "specialization of the bank",
        "specialisation of the bank",
        "was renamed",
        "renamed as",
        "replaced by",
        "transformed into",
    ]

    visible_ifrs_patterns = [
        "[ifrs",
        "ifrs s2 §",
        "ifrs s1 §",
        "§6(a)",
        "§6(b)",
        "§6(a)(v)",
        "§7",
        "§8",
        "§9",
    ]

    # Duplication: detailed decisions should appear only in the dedicated decision subsection.
    board_oversight = _section_text(
        draft,
        "#### Board oversight",
        [
            "#### Management responsibility",
            "#### Climate skills and competencies",
            "#### Remuneration and climate incentives",
            "#### Board and committee decisions during 2024",
            "#### External assurance and controls",
        ]
    ).lower()
    decision_like_dates_in_board_oversight = len(re.findall(
        r"\b\d{1,2}\s+(january|february|march|april|may|june|july|august|september|october|november|december)\b",
        board_oversight
    ))
    duplicated_decisions_in_board_oversight = decision_like_dates_in_board_oversight >= 2

    # Specific strict evidence-boundary checks. These do not prove quality, but catch obvious missing language.
    formal_mandate_language_present = any(
        phrase in text for phrase in [
            "charter", "terms of reference", "formal mandate", "formal governance instrument",
            "does not include the committee charter", "does not include the committee's charter"
        ]
    )
    tradeoff_language_present = any(
        phrase in text for phrase in [
            "trade-off", "tradeoff", "trade-offs", "specific trade-offs", "does not describe specific trade-offs",
            "capital allocation", "risk appetite", "implementation cost", "profitability"
        ]
    )
    skills_process_language_present = any(
        phrase in text for phrase in [
            "skills adequacy", "skills assessment", "skills matrix", "formal board skills", "does not describe a formal board skills"
        ]
    )
    assurance_scope_limitation_present = (
        "scope 1" in text and "scope 2" in text and
        ("financed emissions" in text or "scope 3" in text) and
        any(p in text for p in ["outside the stated assurance scope", "not within the stated assurance scope", "does not extend"])
    )

    hard_fails = {
        "visible_ifrs_references": any(p in text for p in visible_ifrs_patterns),
        "agenda_time_misinterpretation": (
            "agenda time" in text or
            "% of the board's agenda" in text or
            "dedicated to climate" in text
        ),
        "generic_limitation_disclaimer": any(
            phrase in text for phrase in [
                "we acknowledge this limitation",
                "absence of prepared evidence",
                "unable to provide",
                "will strive to provide",
                "this section acknowledges",
            ]
        ),
        "unsupported_committee_evolution": any(p in text for p in unsupported_evolution_patterns),
        "overclaimed_escalation": "significant or material climate risks are escalated" in text,
        "unsafe_assurance_language": any(
            phrase in text for phrase in [
                "moderate level of assurance",
                "moderate assurance",
                "ensuring the reliability",
                "ensures the reliability",
            ]
        ),
        "unsupported_alignment_or_compliance_claim": any(
            phrase in text for phrase in [
                "disclosures are aligned with tcfd",
                "disclosures are aligned with ifrs",
                "aligned with ifrs s2 requirements",
                "fully aligned",
                "compliant with",
                "complies with",
            ]
        ),
        "overstrong_control_language": any(
            phrase in text for phrase in [
                "ensuring regular updates",
                "ensuring transparency",
                "guarantees",
                "fully resilient",
            ]
        ),
        "duplicate_decision_listing": duplicated_decisions_in_board_oversight,
        "missing_formal_mandate_boundary": not formal_mandate_language_present,
        "missing_board_tradeoff_boundary": not tradeoff_language_present,
        "missing_skills_adequacy_boundary": not skills_process_language_present,
        "missing_assurance_scope_limitation": not assurance_scope_limitation_present,
    }

    structure_ok = len(missing_subsections) == 0
    content_ok = not any(hard_fails.values())

    return {
        "passed": structure_ok and content_ok,
        "structure_ok": structure_ok,
        "content_ok": content_ok,
        "missing_subsections": missing_subsections,
        "hard_fails": {k: v for k, v in hard_fails.items() if v},
        "required_fixes": (
            [f"Add missing subsection: {s}" for s in missing_subsections] +
            [f"Fix hard fail: {k}" for k, v in hard_fails.items() if v]
        )
    }


In [9]:
# ── LANGGRAPH NODES ──────────────────────────────────────────

def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)

    draft = call_llm(
        system_prompt=WRITER_SYSTEM,
        user_prompt=prompt,
        temperature=0.2
    )

    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {
        **state,
        "draft": draft.strip(),
        "status": "judging"
    }


def judge_node(state: GovernanceState) -> GovernanceState:
    draft = state["draft"]

    # Step 1: deterministic rule checks
    rules = rule_check(draft)
    print(f"\nRule check: {'PASSED' if rules['passed'] else 'FAILED'}")
    if not rules["passed"]:
        print(f"  Issues: {rules['required_fixes']}")

    if not rules["passed"]:
        # Force a structured judge result from rule failures
        judge_result = {
            "overall_score": 4 if rules["structure_ok"] else 3,
            "evidence_support_score": 5,
            "ifrs_alignment_score": 4,
            "specificity_score": 5,
            "hallucination_risk": "medium",
            "approved": False,
            "checklist": {
                "all_six_subsections_present": rules["structure_ok"],
                "false_count": len(rules["required_fixes"])
            },
            "main_issues": rules["required_fixes"],
            "required_fixes": rules["required_fixes"],
            "rule_check_override": True
        }
        return {
            **state,
            "judge_result": judge_result,
            "status": "judging"
        }

    # Step 2: LLM judge
    judge_prompt = build_judge_prompt(draft, state["evidence"])
    judge_result = call_llm_json(
        system_prompt=JUDGE_SYSTEM,
        user_prompt=judge_prompt
    )

    # Step 3: hard score ceiling enforcement
    false_count = judge_result.get("checklist", {}).get("false_count", 0)
    score = judge_result.get("overall_score", 0)

    ceilings = {0: 10, 1: 8, 2: 7}
    ceiling = ceilings.get(false_count, 6)
    if score > ceiling:
        judge_result["overall_score"] = ceiling
        judge_result["score_ceiling_applied"] = f"Capped at {ceiling} due to {false_count} failed checks"

    # Step 4: approved only if score >= 8 AND false_count == 0
    judge_result["approved"] = (
        judge_result.get("overall_score", 0) >= 8 and
        false_count == 0
    )

    print(f"\nJudge result:")
    print(f"  Score: {judge_result.get('overall_score')}/10")
    print(f"  Approved: {judge_result.get('approved')}")
    print(f"  False checks: {false_count}")
    if judge_result.get("main_issues"):
        print(f"  Issues: {judge_result['main_issues']}")

    return {
        **state,
        "judge_result": judge_result,
        "status": "judging"
    }


def reviser_node(state: GovernanceState) -> GovernanceState:
    return {
        **state,
        "revision_count": state["revision_count"] + 1,
        "status": "drafting"
    }


def finalize_node(state: GovernanceState) -> GovernanceState:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)

    print(f"\n{'='*50}")
    print(f"FINALIZED")
    print(f"  Status: {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10")
    print(f"  Revisions: {state['revision_count']}")
    print(f"{'='*50}")

    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed"
    }


# ── ROUTING ──────────────────────────────────────────────────
def route_after_judge(state: GovernanceState) -> str:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)
    revision_count = state.get("revision_count", 0)
    max_revisions = state.get("max_revisions", 2)

    if approved:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return "revise"

In [10]:
# ── BUILD AND COMPILE GRAPH ───────────────────────────────────
builder = StateGraph(GovernanceState)

builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "writer")
builder.add_edge("finalize", END)

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise":   "reviser",
        "finalize": "finalize",
    }
)

graph = builder.compile()
print("Graph compiled")

Graph compiled


In [11]:
# ── RUN ──────────────────────────────────────────────────────
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  2,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {}
}

print(f"Starting governance generation for: {bank_name}\n")
result = graph.invoke(initial_state)

Starting governance generation for: Eurolux Universal Bank AG


WRITER (initial draft)
Draft length: 683 words

Rule check: FAILED
  Issues: ['Fix hard fail: missing_skills_adequacy_boundary']

WRITER (revision 1)
Draft length: 664 words

Rule check: FAILED
  Issues: ['Fix hard fail: missing_skills_adequacy_boundary']

WRITER (revision 2)
Draft length: 690 words

Rule check: PASSED

Judge result:
  Score: 9/10
  Approved: True
  False checks: 0

FINALIZED
  Status: APPROVED
  Final score: 9/10
  Revisions: 2


In [12]:
# ── OUTPUT ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":         result["status"],
        "final_score":    result["judge_result"].get("overall_score"),
        "revisions":      result["revision_count"],
        "approved":       result["judge_result"].get("approved"),
        "checklist":      result["judge_result"].get("checklist"),
        "issues":         result["judge_result"].get("main_issues"),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved to outputs/governance_BANK01.md")


FINAL JUDGE RESULT
{
  "overall_score": 9,
  "evidence_support_score": 9,
  "ifrs_alignment_score": 9,
  "specificity_score": 9,
  "hallucination_risk": "low",
  "approved": true,
  "checklist": {
    "all_six_subsections_present": true,
    "no_visible_ifrs_refs": true,
    "agenda_pct_correct": true,
    "four_distinct_decisions": true,
    "no_duplicate_decisions": true,
    "yoy_trends_present": true,
    "both_remuneration_figures": true,
    "formal_mandate_or_limitation": true,
    "board_tradeoffs_or_limitation": true,
    "management_process_flow": true,
    "escalation_not_overclaimed": true,
    "no_unsupported_committee_evolution": true,
    "skills_adequacy_or_limitation": true,
    "assurance_explained_safely": true,
    "assurance_scope_limitation": true,
    "no_limitation_disclaimer": true,
    "no_unsupported_strong_claims": true,
    "false_count": 0
  },
  "main_issues": [],
  "required_fixes": []
}

GOVERNANCE SECTION
### Governance

#### Board oversight

Eurolux 